# ENGRAMA V3 — Entrenamiento en TinyStories 🧠⚡

**ENGRAMA** es una arquitectura neuronal autorregresiva **sin atención** (sin $QK^T$, sin matrices $N \times N$, sin softmax temporal), implementada en PyTorch puro según la especificación [ENGRAMA V3](https://github.com/bueormnew/engrama/blob/main/ENGRAMA-V3-Teorica.md).

Este notebook:
1. Instala la librería desde GitHub.
2. Carga TinyStories (Kaggle input / descarga / fallback offline).
3. Entrena en **modo rápido** (`quickstart`) y genera texto.
4. Muestra el **modo experto** (`EngramaConfig`, ablación V2 vs V3).
5. **Verifica la invarianza causal** (forward paralelo == inferencia incremental).

> 🔧 **FAST_MODE = True** (por defecto) ejecuta todo en pocos minutos incluso en CPU. Ponlo a `False` para un entrenamiento más serio (ideal con GPU T4×2).

- Autor: **BUEORM** · Licencia: **AGPL-3.0**


## 1️⃣ Instalación

In [ ]:
# Instalación desde el repositorio oficial (el nombre 'engrama' en PyPI es otro paquete)
!pip install -q git+https://github.com/bueormnew/engrama.git

import torch
import engrama

print('ENGRAMA', engrama.__version__, '| torch', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 2️⃣ Datos: TinyStories

Estrategia en orden: (a) dataset montado en Kaggle, (b) descarga del split de validación de Hugging Face, (c) **fallback sintético offline** para que el notebook siempre sea ejecutable.


In [ ]:
import glob
import os

FAST_MODE = True   # False => entrenamiento largo (recomendado con GPU)

def find_kaggle_tinystories():
    pats = [
        '/kaggle/input/tinystories*/**/*.txt',
        '/kaggle/input/*/TinyStories*.txt',
    ]
    for pat in pats:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return None

def download_tinystories(path='tinystories.txt'):
    url = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
           'TinyStoriesV2-GPT4-valid.txt')
    import urllib.request
    urllib.request.urlretrieve(url, path)
    return path

FALLBACK = (
    'Once upon a time there was a little cat named Lily. Lily liked to play '
    'in the garden. One day she found a red ball. She was very happy and played '
    'all day. Then her friend Tom the dog came and they played together. '
    'At night Lily went home and slept. The end.\n'
)

text = None
src = find_kaggle_tinystories()
if src is not None:
    print('Fuente: Kaggle input ->', src)
    with open(src, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
else:
    try:
        path = download_tinystories()
        print('Fuente: descarga Hugging Face ->', path)
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
    except Exception as exc:  # sin internet: fallback offline
        print('Sin acceso a datos externos (%s). Uso corpus sintético.' % type(exc).__name__)
        text = FALLBACK * 4000

if FAST_MODE:
    text = text[:200_000]        # subconjunto rápido
    n_epochs, size = 2, 'tiny'
else:
    text = text[:20_000_000]     # ~20 MB de cuentos
    n_epochs, size = 3, 'base'

print(f'Corpus: {len(text):,} caracteres | preset={size} epochs={n_epochs}')


## 3️⃣ Modo rápido: `quickstart`

Una sola llamada: tokenizador de caracteres → modelo preset → entrenamiento AdamW con gradient clipping. Devuelve un `QuickRun` con `.generate()`, `.evaluate()` y `.save()`.


In [ ]:
run = engrama.quickstart(
    text,
    size=size,
    epochs=n_epochs,
    batch_size=32,
    sequence_length=128,
    verbose=True,
)
print(run.summary())


### Generación de texto

In [ ]:
for prompt in ['Once upon a time', 'One day', 'The little']:
    out = run.generate(prompt, max_new_tokens=120, temperature=0.8, top_k=40)
    print(f'--- {prompt!r} ---')
    print(out)
    print()


## 4️⃣ Modo experto: `EngramaConfig` y ablación V2 vs V3

Cada modo de arquitectura es configurable; `version` es un preset real (`v1`/`v2` = denso V2, `v3` = factorizado jerárquico) y cualquier modo explícito lo sobrescribe, lo que habilita las ablaciones de la spec (§43–44).


In [ ]:
from engrama import EngramaConfig, EngramaModel

vocab = run.tokenizer.vocab_size
base = dict(vocab_size=vocab, d_model=256, num_cells=8,
            context_length=256, num_candidates=4)

cfg_v3 = EngramaConfig(version='v3', **base)
cfg_v2 = EngramaConfig(version='v2', **base)   # ablación densa
m_v3, m_v2 = EngramaModel(cfg_v3), EngramaModel(cfg_v2)

print(f'Parámetros V3: {m_v3.num_parameters():,}')
print(f'Parámetros V2: {m_v2.num_parameters():,} '
      f'({m_v2.num_parameters()/m_v3.num_parameters():.2f}x más)')
print('Offsets por capa V3:',
      [cfg_v3.get_layer_offsets(l) for l in range(cfg_v3.num_consolidation_layers)])
print('Horizontes de caché jerárquico:', cfg_v3.cache_horizons(),
      '(vs', cfg_v3.context_length * cfg_v3.num_consolidation_layers,
      'estados del caché completo)')


## 5️⃣ Verificación de invarianza causal

Garantía central de ENGRAMA: el forward paralelo de entrenamiento y la inferencia incremental token a token producen **los mismos logits** (error < 1e-4; medido típicamente ~1e-6 en float32).


In [ ]:
model = m_v3.eval()
x = torch.randint(0, vocab, (2, 32))
with torch.no_grad():
    full = model(x)
    for mode in ('full', 'hierarchical'):
        cache = model.get_cache(N_max=32, mode=mode)
        steps = [model.step_forward(x[:, t:t+1], cache, t)[0]
                 for t in range(32)]
        inc = torch.stack(steps, dim=1)
        print(f'caché {mode:13s}: max |diff| = '
              f'{(full - inc).abs().max().item():.2e}')


### Guardar el modelo entrenado

In [ ]:
run.save('./engrama_tinystories')
print('Modelo guardado en ./engrama_tinystories:', os.listdir('./engrama_tinystories'))


## ✅ Conclusiones y notas honestas

- ENGRAMA V3 entrena y genera texto **sin ningún mecanismo de atención**, con un 4–8× menos parámetros que su contraparte densa V2 a igual escala y estado de caché jerárquico mínimo.
- Con FAST_MODE el texto generado es incipiente: es un modelo pequeño entrenado en minutos. Sube `size` a `'base'`/`'large'`, desactiva FAST_MODE y entrena varias épocas en GPU para calidad real.
- Para benchmarks y la verificación completa (68 tests), consulta el repositorio: `tests/`, `docs/VERIFICACION.md` y `benchmarks/KV_RETRIEVAL_REPORT.md`.
- Licencia AGPL-3.0 · Autor: BUEORM · [github.com/bueormnew/engrama](https://github.com/bueormnew/engrama)
